# ABS Real Estate

Peak-to-trough house-price declines, every episode aligned at its peak.

The price series is the long-run spliced mean dwelling price from
`abs_prices.get_house_price_index()`: the current all-dwellings mean price
(ABS 6432.0), the discontinued eight-capitals RPPI and established-house
indexes (ABS 6416.0), and the BIS/REIA index underneath to reach 1970. It is
seasonally adjusted before the drawdowns are measured, because an Original
series would put a seasonal wobble into every episode.

Two views: **nominal**, which is what the commercial trackers plot, and
**real** (CPI deflated), which is what a house actually costs you.

## Python set-up

In [1]:
# analytic imports
import mgplot as mg
import pandas as pd
from pandas import DataFrame, Period, Series

# local imports
from abs_prices import get_house_price_index

# pandas display settings
pd.options.display.max_rows = 999999
pd.options.display.max_columns = 999
pd.options.display.max_colwidth = 100

In [2]:
# Constants
SHOW = False
FILE_TYPE = "png"

source = "ABS 6432.0, 6416.0, 6401.0; BIS/REIA"
LFOOTER = "Australia. Mean dwelling price, seasonally adjusted. "

# Declines shallower than this are not episodes, they are quarterly noise: the
# series carries revisions and a spliced junction, and a one-quarter 1 per cent
# dip says nothing about a downturn.
MIN_DECLINE = 2.0  # per cent, peak to trough

# Line style for an episode that has not yet reached its trough - the series
# has not regained the peak, so the decline may still have further to run.
ONGOING_STYLE, COMPLETE_STYLE = ":", "-"

# Output chart directory (fresh for this notebook).
mg.set_chart_dir("./CHARTS/Real Estate/")
mg.clear_chart_dir()

## Defining a drawdown episode

Each episode starts at a peak - a quarter whose price level is not regained
until some later quarter - and is plotted to its trough, the lowest quarter
before that peak is regained. There is no minimum or maximum length, and no
rule against rallies: a recovery that fails to regain the old peak stays
inside the same episode. That is deliberate, and it is why the real 1989
episode runs 28 quarters and climbs six percentage points in the middle
before falling again.

The alternative - breaking an episode once prices rebound some set amount off
the trough - splits that into the 1989-91 and 1994-95 downturns the commercial
trackers report, but needs a rebound threshold chosen by hand. This high-water
rule has no such free parameter.

One consequence worth knowing: real prices peaked in 1974Q1 and did not regain
that level until 1988Q2, so that whole span is a single episode, and the
1981-83 and 1985-87 falls inside it are not counted separately.

In [3]:
def _next_recovery(series: Series, start: int) -> int:
    """Return the first position after start where the series regains its level.

    Args:
        series: the price level.
        start: the position of the peak.

    Returns:
        The position of the recovery, or len(series) if it never recovers.

    """
    peak = series.iloc[start]
    position = start + 1
    while position < len(series) and series.iloc[position] < peak:
        position += 1
    return position


def _episode_label(peak: Period, trough: Period) -> str:
    """Label an episode by the years it spans, e.g. 1989-96.

    Args:
        peak: the period of the peak.
        trough: the period of the trough.

    Returns:
        The label.

    """
    return f"{peak.year}-{trough.year % 100:02d}"


def drawdown_episodes(series: Series, min_decline: float = MIN_DECLINE) -> tuple[DataFrame, DataFrame]:
    """Find every peak-to-trough decline of at least min_decline per cent.

    An episode runs from a peak (a new high in the series) to the lowest point
    before that peak is regained. The path is expressed as a percentage of the
    peak and re-indexed to quarters since the peak, so the episodes can be
    overlaid on one axis.

    Args:
        series: a house-price level, on a quarterly PeriodIndex.
        min_decline: the smallest peak-to-trough fall to count, in per cent.

    Returns:
        A (paths, summary) tuple - the decline paths, one column per episode
        indexed by quarters since the peak; and a summary table of each
        episode's peak, trough, depth, length and whether it is still running.

    """
    index = series.index
    if not isinstance(index, pd.PeriodIndex):
        raise TypeError(f"Expected a PeriodIndex, got {type(index).__name__}")

    paths: dict[str, Series] = {}
    rows: list[dict[str, object]] = []
    position = 0
    while position < len(series):
        recovery = _next_recovery(series, position)
        leg = series.iloc[position:recovery]
        trough_offset = int(leg.to_numpy().argmin())
        decline = (leg.iloc[trough_offset] / series.iloc[position] - 1) * 100
        if -decline >= min_decline:
            path = (leg.iloc[: trough_offset + 1] / series.iloc[position] - 1) * 100
            label = _episode_label(index[position], leg.index[trough_offset])
            paths[label] = Series(path.to_numpy(), index=range(len(path)))
            rows.append(
                {
                    "Episode": label,
                    "Peak": index[position],
                    "Trough": leg.index[trough_offset],
                    "Decline %": round(decline, 2),
                    "Quarters": trough_offset,
                    "Ongoing": recovery == len(series),
                },
            )
        position = max(recovery, position + 1)

    return DataFrame(paths), DataFrame(rows).set_index("Episode")

## Plot the episodes

In [4]:
def plot_drawdowns(paths: DataFrame, summary: DataFrame, title: str, lfooter: str) -> None:
    """Overlay every decline episode, aligned at its peak.

    Args:
        paths: the decline paths, one column per episode (from drawdown_episodes).
        summary: the matching summary table, used to dot any ongoing episode.
        title: the chart title.
        lfooter: the left footer.

    """
    styles = [
        ONGOING_STYLE if summary.loc[episode, "Ongoing"] else COMPLETE_STYLE
        for episode in paths.columns
    ]
    mg.line_plot_finalise(
        paths,
        title=title,
        xlabel="Quarters since peak",
        ylabel="Per cent from peak",
        style=styles,
        width=1.5,
        annotate=False,
        dropna=True,
        legend={"loc": "lower left", "fontsize": "small", "ncol": 2},
        axhline={"y": 0, "color": "darkgrey", "linewidth": 0.75},
        rfooter=source,
        lfooter=lfooter,
        file_type=FILE_TYPE,
        show=SHOW,
    )

## Nominal declines

In [5]:
nominal, nominal_units, nominal_stype = get_house_price_index(
    extend_bis=True, seasonally_adjusted=True
)
nominal_paths, nominal_summary = drawdown_episodes(nominal)
plot_drawdowns(
    nominal_paths,
    nominal_summary,
    title="House Prices: Drawdowns from Previous Peak",
    lfooter=LFOOTER + "Nominal. ",
)
nominal_summary

,Peak,Trough,Decline %,Quarters,Ongoing
Episode,,,,,
2008-08,2008Q1,2008Q4,-4.97,3,False
2010-11,2010Q2,2011Q4,-2.62,6,False
2018-19,2018Q1,2019Q1,-6.10,4,False
2022-22,2022Q1,2022Q4,-4.67,3,False


## Real declines

The same episodes, after deflating by the headline CPI.

In [6]:
real, real_units, real_stype = get_house_price_index(
    extend_bis=True, real=True, seasonally_adjusted=True
)
real_paths, real_summary = drawdown_episodes(real)
plot_drawdowns(
    real_paths,
    real_summary,
    title="Real House Prices: Drawdowns from Previous Peak",
    lfooter=LFOOTER + f"CPI deflated, {real_units}. ",
)
real_summary

,Peak,Trough,Decline %,Quarters,Ongoing
Episode,,,,,
1974-78,1974Q1,1978Q4,-14.81,19,False
1989-96,1989Q1,1996Q1,-10.03,28,False
2000-00,2000Q2,2000Q3,-3.58,1,False
2003-05,2003Q4,2005Q3,-3.52,7,False
2007-08,2007Q4,2008Q4,-7.48,4,False
2010-12,2010Q2,2012Q3,-7.75,9,False
2017-19,2017Q2,2019Q2,-7.72,8,False
2021-23,2021Q4,2023Q1,-10.74,5,False


## Finished

In [7]:
# watermark
%load_ext watermark
%watermark -u -t -d --iversions --watermark
print("Finished")

Last updated: 2026-08-19 11:14:59

mgplot: 0.2.31
pandas: 3.0.5

Watermark: 2.6.0

Finished
